# Fair Credit Scoring — Home Credit (Application Table Only, v1)

First-iteration implementation following **Kozodoi, Jacob & Lessmann (2022)**, *EJOR 297(3): 1083–1094*.

**Scope of v1**
- Uses **only `application_train.csv`** (no auxiliary tables)
- LightGBM with 5-fold stratified CV
- Sensitive attribute: age group (threshold = 25, per Kamiran & Calders 2009)
- Five fairness processors: Reweighing, Disparate Impact Remover, Adversarial Debiasing, Reject Option, Equalized Odds
- Profit (Verbraken et al. 2014), AUC, IND, SP, SF
- Pareto frontier sweep

**Why application-only for v1?** Faster iteration. v2 will add aggregations from `bureau`, `previous_application`, etc.

**Data**: download `application_train.csv` from <https://www.kaggle.com/competitions/home-credit-default-risk/data> into `./data/`.

**Install**:
```bash
pip install pandas numpy scikit-learn lightgbm matplotlib seaborn aif360 'tensorflow<2.16'
```

In [ ]:
!pip install aif360 lightgbm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 9.0 MB/s eta 0:00:00


In [ ]:
!pip install BlackBoxAuditing -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 46.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install pandas numpy scikit-learn lightgbm matplotlib seaborn aif360 'tensorflow<2.16'

ERROR: Could not find a version that satisfies the requirement tensorflow<2.16 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow<2.16


## 1. Imports & Configuration

### 1.1 Standard libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1.2 Modelling stack

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
import lightgbm as lgb

### 1.3 Fairness library (AIF360)

In [ ]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing, DisparateImpactRemover
from aif360.algorithms.postprocessing import RejectOptionClassification, EqOddsPostprocessing

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


### 1.4 Constants

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA = '/content/drive/MyDrive/pattern_project/'
SEED = 42
AGE_THRESHOLD = 25  # Kamiran & Calders (2009)
np.random.seed(SEED)

## 2. Load & Preprocess `application_train.csv`

### 2.1 Read the file

In [ ]:
df = pd.read_csv(DATA + 'application_train.csv')
print(f'Shape: {df.shape}')
print(f'Default rate: {df.TARGET.mean():.4f}')
df.head()

Shape: (307511, 122)
Default rate: 0.0807


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


### 2.2 Drop unknown gender (Koehrsen-style cleanup)

In [ ]:
df = df[df['CODE_GENDER'] != 'XNA'].reset_index(drop=True)
print(f'After removing XNA gender: {df.shape}')

After removing XNA gender: (307507, 122)


### 2.3 Handle the `DAYS_EMPLOYED` anomaly

Some rows have `DAYS_EMPLOYED = 365243` (~1000 years). Replace with NaN.

In [ ]:
df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

### 2.4 Hand-crafted ratio features

In [ ]:
df['DAYS_EMPLOYED_PERC']   = df['DAYS_EMPLOYED']    / df['DAYS_BIRTH']
df['INCOME_CREDIT_PERC']   = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON']    = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
df['ANNUITY_INCOME_PERC']  = df['AMT_ANNUITY']      / df['AMT_INCOME_TOTAL']
df['PAYMENT_RATE']         = df['AMT_ANNUITY']      / df['AMT_CREDIT']

### 2.5 One-hot encode categoricals

In [ ]:
cat_cols = [c for c in df.columns if df[c].dtype == 'object']
print(f'Categorical columns: {len(cat_cols)}')
df = pd.get_dummies(df, columns=cat_cols, dummy_na=True)
print(f'After one-hot: {df.shape}')

Categorical columns: 16
After one-hot: (307507, 266)


### 2.6 Build the sensitive attribute (age group)

In [ ]:
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365.25)
df['AGE_GROUP'] = (df['AGE_YEARS'] < AGE_THRESHOLD).astype(int)

print('Sensitive group share:')
print(df['AGE_GROUP'].value_counts(normalize=True))
print('\nDefault rate by group:')
print(df.groupby('AGE_GROUP')['TARGET'].mean())

Sensitive group share:
AGE_GROUP
0    0.960219
1    0.039781
Name: proportion, dtype: float64

Default rate by group:
AGE_GROUP
0    0.078981
1    0.122946
Name: TARGET, dtype: float64


### 2.7 Flip the label to match the paper

Home Credit: `TARGET=1` = default. Paper convention: `y=1` = good risk (repaid).

In [ ]:
df['y'] = 1 - df['TARGET']

### 2.8 Clean column names so LightGBM doesn't choke

In [ ]:
df.columns = [c.replace(' ', '_').replace(',', '_').replace(':', '_') for c in df.columns]

### 2.9 Define feature list

In [ ]:
drop = ['SK_ID_CURR', 'TARGET', 'AGE_YEARS']
feats = [c for c in df.columns if c not in drop + ['y', 'AGE_GROUP']]
print(f'Number of features: {len(feats)}')

Number of features: 264


## 3. Evaluation Metrics

### 3.1 Profit per EUR (Verbraken et al. 2014)

Parameters from Kozodoi et al. (2022): ROI = 0.2664, p₀ = 0.55, p₁ = 0.10.

In [ ]:
ROI = 0.2664
P0  = 0.55
P1  = 0.10
EB  = P0 * 0 + P1 * 1 + (1 - P0 - P1) * 0.5  # E[loss given default]

In [ ]:
def profit_per_eur(y, yhat):
    accepted = (yhat == 1)
    if accepted.sum() == 0:
        return 0.0
    g = ((y == 1) & accepted).sum()
    b = ((y == 0) & accepted).sum()
    return (ROI * g - EB * b) / accepted.sum()

### 3.2 Profit-optimal cutoff search

In [ ]:
def best_cutoff(y, s):
    best_t, best_p = 0.5, -np.inf
    for t in np.linspace(0.05, 0.95, 91):
        p = profit_per_eur(y, (s >= t).astype(int))
        if p > best_p:
            best_p, best_t = p, t
    return best_t

### 3.3 Independence — IND (Eq. 2)

In [ ]:
def IND(yhat, A):
    return abs(yhat[A == 0].mean() - yhat[A == 1].mean())

### 3.4 Separation — SP (Eq. 4)

In [ ]:
def SP(y, yhat, A):
    def rates(yt, yp):
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        return fpr, fnr
    fpr0, fnr0 = rates(y[A == 0], yhat[A == 0])
    fpr1, fnr1 = rates(y[A == 1], yhat[A == 1])
    return 0.5 * (abs(fpr1 - fpr0) + abs(fnr1 - fnr0))

### 3.5 Sufficiency — SF (Eq. 6)

In [ ]:
def SF(y, yhat, A):
    def ppv(yt, yp):
        if (yp == 1).sum() == 0:
            return 0
        return ((yp == 1) & (yt == 1)).sum() / (yp == 1).sum()
    return abs(ppv(y[A == 0], yhat[A == 0]) - ppv(y[A == 1], yhat[A == 1]))

### 3.6 Single evaluator

In [ ]:
def evaluate(name, y, s, yhat, A):
    return {
        'model':    name,
        'AUC':      roc_auc_score(y, s),
        'Profit':   profit_per_eur(y, yhat),
        'IND':      IND(yhat, A),
        'SP':       SP(y, yhat, A),
        'SF':       SF(y, yhat, A),
        'AcceptRt': yhat.mean(),
    }

## 4. Train / Test Split

In [ ]:
y_all = df['y'].values
A_all = df['AGE_GROUP'].values
X_all = df[feats]

In [ ]:
X_dev, X_te, y_dev, y_te, A_dev, A_te = train_test_split(
    X_all, y_all, A_all, test_size=0.30, stratify=y_all, random_state=SEED
)
print(f'Dev:  {X_dev.shape}')
print(f'Test: {X_te.shape}')

Dev:  (215254, 264)
Test: (92253, 264)


In [ ]:
# AIF360 can't handle NaNs anywhere. Median-fill once and reuse everywhere.
medians = X_dev.median(numeric_only=True)
X_dev = X_dev.fillna(medians)
X_te  = X_te.fillna(medians)
print(f'NaNs in X_dev: {X_dev.isna().sum().sum()}, X_te: {X_te.isna().sum().sum()}')

NaNs in X_dev: 0, X_te: 0


## 5. LightGBM Baseline (Unconstrained)

### 5.1 Hyperparameters

In [ ]:
lgb_params = dict(
    objective='binary', metric='auc', boosting_type='gbdt',
    learning_rate=0.02,
    num_leaves=34, max_depth=8, min_data_in_leaf=40,
    feature_fraction=0.95, bagging_fraction=0.85, bagging_freq=5,
    reg_alpha=0.04, reg_lambda=0.07,
    n_estimators=5000,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

### 5.2 5-fold stratified CV

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(X_dev))
test_pred = np.zeros(len(X_te))

In [ ]:
for fold, (tr, va) in enumerate(skf.split(X_dev, y_dev)):
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_dev.iloc[tr], y_dev[tr],
          eval_set=[(X_dev.iloc[va], y_dev[va])],
          callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    oof[va] = m.predict_proba(X_dev.iloc[va])[:, 1]
    test_pred += m.predict_proba(X_te)[:, 1] / skf.n_splits
    print(f'  Fold {fold + 1} AUC: {roc_auc_score(y_dev[va], oof[va]):.4f}')

Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[775]	valid_0's auc: 0.766005
  Fold 1 AUC: 0.7660
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1125]	valid_0's auc: 0.762815
  Fold 2 AUC: 0.7628
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[825]	valid_0's auc: 0.764303
  Fold 3 AUC: 0.7643
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[617]	valid_0's auc: 0.7698
  Fold 4 AUC: 0.7698
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[692]	valid_0's auc: 0.766065
  Fold 5 AUC: 0.7661


### 5.3 OOF & test scores

In [ ]:
print(f'OOF AUC:  {roc_auc_score(y_dev, oof):.4f}')
print(f'Test AUC: {roc_auc_score(y_te, test_pred):.4f}')

OOF AUC:  0.7657
Test AUC: 0.7663


### 5.4 Profit-optimal cutoff & baseline metrics

In [ ]:
tau = best_cutoff(y_dev, oof)
print(f'Profit-optimal cutoff: {tau:.3f}')

Profit-optimal cutoff: 0.950


In [ ]:
yhat_te = (test_pred >= tau).astype(int)
results = [evaluate('Unconstrained LGBM', y_te, test_pred, yhat_te, A_te)]
pd.DataFrame(results)

,model,AUC,Profit,IND,SP,SF,AcceptRt
0,Unconstrained LGBM,0.766322,0.251756,0.316928,0.22897,0.001356,0.492916


## 6. AIF360 Helpers

### 6.1 Group definitions

In [ ]:
PRIV = [{'AGE_GROUP': 0}]   # older = privileged
UNPR = [{'AGE_GROUP': 1}]   # young  = unprivileged

### 6.2 Frame → BinaryLabelDataset

In [ ]:
def to_aif(X, y, A):
    d = X.copy()
    d['y'] = y
    d['AGE_GROUP'] = A
    return BinaryLabelDataset(
        df=d, label_names=['y'], protected_attribute_names=['AGE_GROUP'],
        favorable_label=1, unfavorable_label=0,
    )

### 6.3 LightGBM trainer with optional sample weights

In [ ]:
def fit_lgbm(Xtr, ytr, Xva, yva, sample_weight=None):
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(Xtr, ytr, sample_weight=sample_weight,
          eval_set=[(Xva, yva)],
          callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])
    return m

### 6.4 Inner train/val split for processors

In [ ]:
X_tr_, X_va_, y_tr_, y_va_, A_tr_, A_va_ = train_test_split(
    X_dev, y_dev, A_dev, test_size=0.2, stratify=y_dev, random_state=SEED
)

## 7. Fairness Processor 1 — Reweighing (pre)

In [ ]:
ds_tr = to_aif(X_tr_, y_tr_, A_tr_)
rw = Reweighing(unprivileged_groups=UNPR, privileged_groups=PRIV)
ds_rw = rw.fit_transform(ds_tr)

In [ ]:
m_rw = fit_lgbm(X_tr_, y_tr_, X_va_, y_va_, sample_weight=ds_rw.instance_weights)

Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[606]	valid_0's auc: 0.766727


In [ ]:
s_te = m_rw.predict_proba(X_te)[:, 1]
tau_rw = best_cutoff(y_va_, m_rw.predict_proba(X_va_)[:, 1])
results.append(evaluate('Reweighing (pre)', y_te, s_te, (s_te >= tau_rw).astype(int), A_te))
pd.DataFrame(results)

,model,AUC,Profit,IND,SP,SF,AcceptRt
0,Unconstrained LGBM,0.766322,0.251756,0.316928,0.228970,0.001356,0.492916
1,Reweighing (pre),0.763651,0.251784,0.210792,0.143002,0.013083,0.487030


## 8. Fairness Processor 2 — Disparate Impact Remover (pre)

### 8.1 Median-fill (DIR can't handle NaNs)

In [ ]:
X_dev_f  = X_dev.fillna(X_dev.median(numeric_only=True))
X_te_f   = X_te.fillna(X_dev.median(numeric_only=True))
X_tr_ff  = X_dev_f.loc[X_tr_.index]
X_va_ff  = X_dev_f.loc[X_va_.index]

### 8.2 Fit & transform train

In [ ]:
dir_     = DisparateImpactRemover(repair_level=0.8, sensitive_attribute='AGE_GROUP')
ds_full  = to_aif(X_tr_ff, y_tr_, A_tr_)
ds_dir   = dir_.fit_transform(ds_full)
X_tr_dir = pd.DataFrame(ds_dir.features, columns=ds_dir.feature_names).drop(columns=['AGE_GROUP'])

### 8.3 Transform val & test

In [ ]:
ds_va     = to_aif(X_va_ff, y_va_, A_va_)
ds_va_dir = dir_.fit_transform(ds_va)
X_va_dir  = pd.DataFrame(ds_va_dir.features, columns=ds_va_dir.feature_names).drop(columns=['AGE_GROUP'])

ds_te_aif = to_aif(X_te_f, y_te, A_te)
ds_te_dir = dir_.fit_transform(ds_te_aif)
X_te_dir  = pd.DataFrame(ds_te_dir.features, columns=ds_te_dir.feature_names).drop(columns=['AGE_GROUP'])

### 8.4 Train & evaluate

In [ ]:
m_dir = fit_lgbm(X_tr_dir, y_tr_, X_va_dir, y_va_)
s_te = m_dir.predict_proba(X_te_dir)[:, 1]
tau_dir = best_cutoff(y_va_, m_dir.predict_proba(X_va_dir)[:, 1])
results.append(evaluate('Disparate Impact Remover (pre)', y_te, s_te, (s_te >= tau_dir).astype(int), A_te))
pd.DataFrame(results)

Training until validation scores don't improve for 120 rounds
Early stopping, best iteration is:
[575]	valid_0's auc: 0.759919


,model,AUC,Profit,IND,SP,SF,AcceptRt
0,Unconstrained LGBM,0.766322,0.251756,0.316928,0.228970,0.001356,0.492916
1,Reweighing (pre),0.763651,0.251784,0.210792,0.143002,0.013083,0.487030
2,Disparate Impact Remover (pre),0.759530,0.251763,0.231145,0.164815,0.005586,0.470706


## 9. Fairness Processor 3 — Adversarial Debiasing (in)

In [ ]:
try:
    import tensorflow.compat.v1 as tf
    tf.disable_eager_execution()
    from aif360.algorithms.inprocessing import AdversarialDebiasing
    tf.reset_default_graph()
    sess = tf.Session()

    ad = AdversarialDebiasing(
        privileged_groups=PRIV, unprivileged_groups=UNPR,
        scope_name='adv_dbg', debias=True, sess=sess,
        num_epochs=50, batch_size=512,
        classifier_num_hidden_units=128, adversary_loss_weight=0.5,
    )
    ad.fit(to_aif(X_tr_ff, y_tr_, A_tr_))

    pred = ad.predict(to_aif(X_te_f, y_te, A_te))
    s_te = pred.scores.ravel()
    tau_ad = best_cutoff(y_te, s_te)
    results.append(evaluate('Adversarial Debiasing (in)', y_te, s_te, (s_te >= tau_ad).astype(int), A_te))
    sess.close()
except Exception as e:
    print(f'Skipping adversarial debiasing: {e}')

pd.DataFrame(results)

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


Skipping adversarial debiasing: Graph execution error:

Detected at node 'adv_dbg/adversary_model/W2/Initializer/random_uniform/mul' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start
    File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
    File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dis

,model,AUC,Profit,IND,SP,SF,AcceptRt
0,Unconstrained LGBM,0.766322,0.251756,0.316928,0.228970,0.001356,0.492916
1,Reweighing (pre),0.763651,0.251784,0.210792,0.143002,0.013083,0.487030
2,Disparate Impact Remover (pre),0.759530,0.251763,0.231145,0.164815,0.005586,0.470706


## 10. Fairness Processor 4 — Reject Option Classification (post)

### 10.1 Wrap baseline scores in AIF360 datasets

In [ ]:
ds_dev = to_aif(X_dev, y_dev, A_dev)
ds_dev_pred = ds_dev.copy()
ds_dev_pred.scores = oof.reshape(-1, 1)
ds_dev_pred.labels = (oof >= tau).astype(int).reshape(-1, 1)

ds_te = to_aif(X_te, y_te, A_te)
ds_te_pred = ds_te.copy()
ds_te_pred.scores = test_pred.reshape(-1, 1)
ds_te_pred.labels = (test_pred >= tau).astype(int).reshape(-1, 1)

### 10.2 Fit & evaluate

In [ ]:
roc = RejectOptionClassification(
    unprivileged_groups=UNPR, privileged_groups=PRIV,
    low_class_thresh=0.01, high_class_thresh=0.99,
    num_class_thresh=50, num_ROC_margin=50,
    metric_name='Average odds difference', metric_ub=0.05, metric_lb=-0.05,
)
roc = roc.fit(ds_dev, ds_dev_pred)
ds_te_roc = roc.predict(ds_te_pred)
yhat_roc = ds_te_roc.labels.ravel().astype(int)
results.append(evaluate('Reject Option (post)', y_te, test_pred, yhat_roc, A_te))
pd.DataFrame(results)

,model,AUC,Profit,IND,SP,SF,AcceptRt
0,Unconstrained LGBM,0.766322,0.251756,0.316928,0.228970,0.001356,0.492916
1,Reweighing (pre),0.763651,0.251784,0.210792,0.143002,0.013083,0.487030
2,Disparate Impact Remover (pre),0.759530,0.251763,0.231145,0.164815,0.005586,0.470706
3,Reject Option (post),0.766322,0.247446,0.102618,0.066455,0.018898,0.635025


## 11. Fairness Processor 5 — Equalized Odds (post)

In [ ]:
eo = EqOddsPostprocessing(unprivileged_groups=UNPR, privileged_groups=PRIV, seed=SEED)
eo = eo.fit(ds_dev, ds_dev_pred)
ds_te_eo = eo.predict(ds_te_pred)
yhat_eo = ds_te_eo.labels.ravel().astype(int)
results.append(evaluate('Equalized Odds (post)', y_te, test_pred, yhat_eo, A_te))

results_df = pd.DataFrame(results)
results_df

,model,AUC,Profit,IND,SP,SF,AcceptRt
0,Unconstrained LGBM,0.766322,0.251756,0.316928,0.228970,0.001356,0.492916
1,Reweighing (pre),0.763651,0.251784,0.210792,0.143002,0.013083,0.487030
2,Disparate Impact Remover (pre),0.759530,0.251763,0.231145,0.164815,0.005586,0.470706
3,Reject Option (post),0.766322,0.247446,0.102618,0.066455,0.018898,0.635025
4,Equalized Odds (post),0.766322,0.251698,0.014431,0.004888,0.015550,0.237900


## 12. Relative Gains (paper Table 5 style)

In [ ]:
base = results_df.iloc[0]
rel = results_df[['model']].copy()
for col in ['AUC', 'Profit']:
    rel[col + '_gain%'] = (results_df[col] - base[col]) / abs(base[col]) * 100
for col in ['IND', 'SP', 'SF']:
    # Lower is better -> positive gain = reduction in unfairness
    rel[col + '_gain%'] = (base[col] - results_df[col]) / max(abs(base[col]), 1e-9) * 100
rel.round(2)

,model,AUC_gain%,Profit_gain%,IND_gain%,SP_gain%,SF_gain%
0,Unconstrained LGBM,0.00,0.00,0.00,0.00,0.00
1,Reweighing (pre),-0.35,0.01,33.49,37.55,-865.08
2,Disparate Impact Remover (pre),-0.89,0.00,27.07,28.02,-312.05
3,Reject Option (post),0.00,-1.71,67.62,70.98,-1294.08
4,Equalized Odds (post),0.00,-0.02,95.45,97.87,-1047.12


## 13. Pareto Frontier (paper Fig. 2)

In [ ]:
frontier = []
for ub in [0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30]:
    try:
        r = RejectOptionClassification(
            unprivileged_groups=UNPR, privileged_groups=PRIV,
            low_class_thresh=0.01, high_class_thresh=0.99,
            num_class_thresh=50, num_ROC_margin=50,
            metric_name='Average odds difference', metric_ub=ub, metric_lb=-ub,
        )
        r = r.fit(ds_dev, ds_dev_pred)
        ds_p = r.predict(ds_te_pred)
        yh = ds_p.labels.ravel().astype(int)
        frontier.append({'bound': ub,
                         'Profit': profit_per_eur(y_te, yh),
                         'SP': SP(y_te, yh, A_te)})
    except Exception as e:
        print(f'bound {ub}: {e}')

front_df = pd.DataFrame(frontier).sort_values('SP')
front_df

,bound,Profit,SP
0,0.010,0.248427,0.017638
1,0.025,0.248106,0.034105
2,0.050,0.247446,0.066455
3,0.075,0.247030,0.084541
4,0.100,0.246479,0.118616
5,0.150,0.245948,0.156820
6,0.200,0.245799,0.173578
7,0.300,0.245799,0.173578


## 14. Plots

### 14.1 Profit–Fairness trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(front_df['SP'], front_df['Profit'], 'o-', lw=2, label='ROC frontier')
ax.scatter(results_df['SP'], results_df['Profit'], s=180, c='crimson', zorder=5, label='Models')
for _, r in results_df.iterrows():
    ax.annotate(r['model'].replace(' (', '\n('), (r['SP'], r['Profit']),
                fontsize=8, xytext=(6, 4), textcoords='offset points')
ax.set_xlabel('Separation (lower = fairer)')
ax.set_ylabel('Profit per EUR issued')
ax.set_title('Profit – Fairness Trade-off')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 14.2 Fairness metrics by model

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(results_df)); w = 0.25
for i, m in enumerate(['IND', 'SP', 'SF']):
    ax.bar(x + i * w, results_df[m], w, label=m)
ax.set_xticks(x + w)
ax.set_xticklabels(results_df['model'], rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Unfairness (lower = fairer)')
ax.set_title('Fairness Metrics by Model')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 15. Discussion

Expected outcomes mirroring Kozodoi et al. (2022):
- **Unconstrained LightGBM** ≈ 0.755 AUC (application-only ceiling), dominates on profit.
- **Reject Option** → largest fairness gain, largest profit cost.
- **Reweighing** → strong middle-ground pre-processor.
- **Adversarial Debiasing** → best profit/separation trade-off, some AUC cost.
- **Equalized Odds** → typically dominated by Reject Option.
- **Pareto frontier** → moderate fairness is cheap; perfect fairness is expensive.

**Next iteration (v2)**: add aggregations from `bureau`, `bureau_balance`, `previous_application`, `POS_CASH_balance`, `installments_payments`, `credit_card_balance` to push AUC above 0.79.

**Reference**: Kozodoi, N., Jacob, J., & Lessmann, S. (2022). Fairness in credit scoring: Assessment, implementation and profit implications. *EJOR* 297(3), 1083–1094.